In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

In [2]:
# 1. Load your data
input_file= "Exoplanet 6059-b.xlsx"   # change to your filename
output_file= "Exoplanet 6059-b_filled_from_pscomppars_Finale.xlsx"

df=pd.read_excel(input_file)

In [3]:
# 2. Pull richer exoplanet data from NASA Exoplanet Archive
# Source table: pscomppars
query="""
select
    pl_name,
    hostname,
    pl_orbper,
    pl_orbsmax,
    pl_rade,
    pl_bmasse,
    pl_eqt,
    st_spectype,
    st_teff,
    st_rad,
    st_mass,
    st_logg,
    sy_dist
from pscomppars
"""

url=(
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?"
    "query="+query.replace("\n", " ").replace(" ", "+")+
    "&format=csv"
)

nasa=pd.read_csv(url)

In [4]:
# 3. Rename NASA columns to match your spreadsheet headers
rename_map={
    "pl_name": "Planet Name",
    "hostname": "Host Name",
    "pl_orbper": "Orbital Period [Days]",
    "pl_orbsmax": "Orbit Semi-Major Axis [au]",
    "pl_rade": "Planet Radius [Earth Radius]",
    "pl_bmasse": "Planet Mass or Mass*sin(i) [Earth Mass]",
    "pl_eqt": "Equilibrium Temperature [K]",
    "st_spectype": "Star Spectral Type",
    "st_teff": "Stellar Effective Temperature [K]",
    "st_rad": "Stellar Radius [Solar Radius]",
    "st_mass": "Stellar Mass [Solar mass]",
    "st_logg": "Stellar Surface Gravity [log10(cm/s**2)]",
    "sy_dist": "Distance [pc]"
}
nasa=nasa.rename(columns=rename_map)

# 4. Merge by planet name
merged=df.merge(
    nasa,
    on="Planet Name",
    how="left",
    suffixes=("", "_NASA")
)


In [5]:
# 5. Fill only missing values in the original file
columns_to_fill=[
    "Orbital Period [Days]",
    "Orbit Semi-Major Axis [au]",
    "Planet Radius [Earth Radius]",
    "Planet Mass or Mass*sin(i) [Earth Mass]",
    "Equilibrium Temperature [K]",
    "Star Spectral Type",
    "Stellar Effective Temperature [K]",
    "Stellar Radius [Solar Radius]",
    "Stellar Mass [Solar mass]",
    "Stellar Surface Gravity [log10(cm/s**2)]",
    "Distance [pc]"
]

fill_counts={}

for col in columns_to_fill:
    nasa_col=f"{col}_NASA"
    before_missing=merged[col].isna().sum()
    merged[col]=merged[col].fillna(merged[nasa_col])
    after_missing=merged[col].isna().sum()
    fill_counts[col]=before_missing - after_missing

# Keep only the original columns
filled_df=merged[df.columns]

# 6. Save to Excel first

filled_df.to_excel(output_file, index=False)

In [6]:
# 7. Highlight cells that were filled from NASA
wb=load_workbook(output_file)
ws=wb.active
green_fill=PatternFill(fill_type="solid", fgColor="C6EFCE")

# Map headers to column numbers
header_map={cell.value: cell.column for cell in ws[1]}

for row_idx in range(2, len(merged) + 2):
    for col in columns_to_fill:
        orig_val=df.iloc[row_idx - 2][col]
        new_val=filled_df.iloc[row_idx - 2][col]

        if pd.isna(orig_val) and pd.notna(new_val):
            excel_col=header_map[col]
            ws.cell(row=row_idx, column=excel_col).fill = green_fill

# 8. Add a summary sheet

summary=wb.create_sheet("Fill Summary")
summary["A1"]="Source"
summary["B1"]="NASA Exoplanet Archive - pscomppars via TAP"

summary["A3"]="Column"
summary["B3"]="Values Filled"

r=4
for col, count in fill_counts.items():
    summary[f"A{r}"]=col
    summary[f"B{r}"]=count
    r+=1

summary[f"A{r+1}"]="Query"
summary[f"B{r+1}"]=query.strip()

wb.save(output_file)

print("Done.")
print("Saved:", output_file)
print("\nFill counts:")
for k, v in fill_counts.items():
    print(f"{k}: {v}")

Done.
Saved: Exoplanet 6059-b_filled_from_pscomppars_Finale.xlsx

Fill counts:
Orbital Period [Days]: 12
Orbit Semi-Major Axis [au]: 1984
Planet Radius [Earth Radius]: 1475
Planet Mass or Mass*sin(i) [Earth Mass]: 3059
Equilibrium Temperature [K]: 2878
Star Spectral Type: 934
Stellar Effective Temperature [K]: 410
Stellar Radius [Solar Radius]: 463
Stellar Mass [Solar mass]: 750
Stellar Surface Gravity [log10(cm/s**2)]: 733
Distance [pc]: 95


In [7]:
# After Fill
sf=pd.read_excel("Exoplanet 6059-b_filled_from_pscomppars_Finale.xlsx")
sf.isna().sum()

Planet Name                                    0
Host Name                                      0
Default Flag                                   0
Number of Stars                                0
Number of Planets                              0
Discovery Method                               0
Soltype                                        0
Controversial Flag                             0
Orbital Period [Days]                        320
Orbit Semi-Major Axis [au]                   313
Planet Radius [Earth Radius]                  44
Planet Mass or Mass*sin(i) [Earth Mass]       26
Equilibrium Temperature [K]                 1504
Star Spectral Type                          3815
Stellar Effective Temperature [K]            281
Stellar Radius [Solar Radius]                299
Stellar Mass [Solar mass]                      5
Stellar Surface Gravity [log10(cm/s**2)]     305
Distance [pc]                                 27
dtype: int64

In [8]:
# Before Fill
df.isna().sum()

Planet Name                                    0
Host Name                                      0
Default Flag                                   0
Number of Stars                                0
Number of Planets                              0
Discovery Method                               0
Soltype                                        0
Controversial Flag                             0
Orbital Period [Days]                        332
Orbit Semi-Major Axis [au]                  2297
Planet Radius [Earth Radius]                1519
Planet Mass or Mass*sin(i) [Earth Mass]     3085
Equilibrium Temperature [K]                 4382
Star Spectral Type                          4749
Stellar Effective Temperature [K]            691
Stellar Radius [Solar Radius]                762
Stellar Mass [Solar mass]                    755
Stellar Surface Gravity [log10(cm/s**2)]    1038
Distance [pc]                                122
dtype: int64